# 🎨 ComfyUI Colab — Batch Workflow Runner

A clean, modular setup for running ComfyUI in Google Colab with a built-in batch processor.

**Run cells top to bottom.** Code is hidden by default — expand to edit.

---
| Step | What it does |
|------|-------------|
| 1 · Mount Drive | Connects Google Drive |
| 2 · Install | ComfyUI + speed stack (xformers, triton, sageattention) |
| 3 · Custom Nodes | Install nodes from GitHub URLs |
| 4 · Models | Download models via aria2c |
| 5 · Launch | Starts ComfyUI + batch watcher |


In [1]:
# @title 📂 1 · Mount Google Drive { display-mode: "form" }
# @markdown Mounts Google Drive (My Drive + Shared Drives). Run this first before any other cell.
from google.colab import drive
import os

print("📂 Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Verify access
if os.path.exists("/content/drive/Shareddrives"):
    print("✅ Shared Drives detected.")
elif os.path.exists("/content/drive/Shared drives"):
    print("✅ Shared Drives detected (alternate path).")
else:
    print("ℹ️  Only MyDrive detected.")

print("✅ Drive mount complete.")


📂 Connecting to Google Drive...
Mounted at /content/drive
✅ Shared Drives detected.
✅ Drive mount complete.


In [2]:
# @title ⚙️ Install ComfyUI & Dependencies { display-mode: "form" }
# @markdown ## 2 · Install ComfyUI & Dependencies
import os
from pathlib import Path

WORKSPACE = "/content/ComfyUI"

# ── 0. Install uv ─────────────────────────────────────────────────
print("📦 Installing uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
print("✅ uv ready\n")

# ── 1. Clone / update ComfyUI ─────────────────────────────────────
if not os.path.exists(WORKSPACE):
    print("📥 Cloning ComfyUI...")
    !git clone -q https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
    print("✅ Cloned\n")
else:
    print("✅ ComfyUI exists, pulling updates...")
    !cd {WORKSPACE} && git pull -q
    print("✅ Updated\n")

%cd {WORKSPACE}

# ── 2. PyTorch cu128 (must match system CUDA 12.8 for custom_rasterizer build) ──
print("🔥 Installing PyTorch cu128 to match system CUDA...")
!uv pip install --system \
    "torch" \
    "torchvision" \
    "torchaudio" \
    --index-url https://download.pytorch.org/whl/cu128

import torch
print(f"  torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}\n")

# ── 3. Speed stack ────────────────────────────────────────────────
print("🚀 Installing speed stack...")
!uv pip install --system --upgrade xformers sageattention

# ── 4. ComfyUI requirements ───────────────────────────────────────
print("📦 Installing ComfyUI requirements...")
!uv pip install --system -r requirements.txt

# ── 5. Core dependencies ──────────────────────────────────────────
print("📚 Installing core dependencies...")
!uv pip install --system \
    accelerate einops diffusers \
    "safetensors>=0.4.2" \
    aiohttp pyyaml Pillow scipy tqdm psutil \
    "tokenizers>=0.13.3" sentencepiece soundfile \
    "kornia>=0.7.1" spandrel torchsde \
    av albumentations opencv-python \
    onnxruntime-gpu color-matcher \
    comfy_aimdo comfy-kitchen \
    pynanoinstantmeshes

# ── 6. Transformers / HuggingFace ────────────────────────────────
print("🤗 Installing transformers & huggingface-hub (latest stable)...")
!uv pip install --system --upgrade \
    "transformers>=4.45.0" \
    "huggingface-hub>=0.23.0" \
    hf_transfer

# ── 7. ninja (required for fast CUDA extension builds) ───────────
print("🥷 Installing ninja...")
!uv pip install --system ninja
print("✅ ninja ready\n")

# ── 8. CUDA memory optimization ──────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅ CUDA memory config set\n")

# ── 9. ComfyUI Manager ────────────────────────────────────────────
manager_path = f"{WORKSPACE}/custom_nodes/ComfyUI-Manager"
if not os.path.exists(manager_path):
    print("📥 Installing ComfyUI Manager...")
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager {manager_path}
else:
    print("🔄 Updating ComfyUI Manager...")
    !cd {manager_path} && git pull -q
print("✅ ComfyUI Manager ready\n")

# ── 10. Verify ────────────────────────────────────────────────────
import importlib.metadata as meta
print("\n📋 Key package versions:")
for pkg in ["torch", "xformers", "transformers", "huggingface-hub", "safetensors", "ninja"]:
    try:
        print(f"  ✅ {pkg}: {meta.version(pkg)}")
    except Exception:
        print(f"  ❌ {pkg}: not found")

print("\n🎉 Installation complete! Run the next cell.")

📦 Installing uv...
downloading uv 0.11.3 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
✅ uv ready

📥 Cloning ComfyUI...
✅ Cloned

/content/ComfyUI
🔥 Installing PyTorch cu128 to match system CUDA...
Using Python 3.12.13 environment at: /usr
Checked 3 packages in 124ms
  torch: 2.10.0+cu128 | CUDA available: True

🚀 Installing speed stack...
Using Python 3.12.13 environment at: /usr
Resolved 32 packages in 399ms
Prepared 23 packages in 15.04s
Uninstalled 6 packages in 568ms
Installed 23 packages in 236ms
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - fsspec==2025.3.0
 + fsspec==2026.3.0
 - numpy==2.0.2
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cuspar

In [3]:
# @title 🔧 3 · Custom Nodes { display-mode: "form" }
# @markdown Add GitHub repos to `CUSTOM_NODES` below and uncomment to install.
# @markdown Re-running is safe — existing nodes are skipped automatically.
import os, shutil

# ─── CONFIGURATION ───
WORKSPACE        = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{WORKSPACE}/custom_nodes"

# Add any GitHub URL here. Format: ("Folder_Name", "URL")
CUSTOM_NODES = [
    ("ComfyUI-Manager",               "https://github.com/ltdrdata/ComfyUI-Manager"),
    ("rgthree-comfy",                  "https://github.com/rgthree/rgthree-comfy"),
    ("ComfyUI-Hunyuan3DWrapper",       "https://github.com/kijai/ComfyUI-Hunyuan3DWrapper"),
    ("ComfyUI-Hunyuan3d-2-1",         "https://github.com/visualbruno/ComfyUI-Hunyuan3d-2-1"),
    ("ComfyUI_IPAdapter_plus",         "https://github.com/cubiq/ComfyUI_IPAdapter_plus"),
    ("ComfyUI-Impact-Pack",            "https://github.com/ltdrdata/ComfyUI-Impact-Pack"),
    ("ComfyUI-Impact-Subpack",         "https://github.com/ltdrdata/ComfyUI-Impact-Subpack"),
    ("ComfyUI_UltimateSDUpscale",      "https://github.com/ssitu/ComfyUI_UltimateSDUpscale"),
    ("ComfyUI_InvSR",                  "https://github.com/yuvraj108c/ComfyUI_InvSR"),
    ("ComfyUI_essentials",             "https://github.com/cubiq/ComfyUI_essentials"),
    ("ComfyUI-KJNodes",                "https://github.com/kijai/ComfyUI-KJNodes"),
    ("comfyui_controlnet_aux",         "https://github.com/Fannovel16/comfyui_controlnet_aux"),
    ("ComfyUI-Inpaint-CropAndStitch",  "https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch"),
    ("ComfyUI-RMBG",                   "https://github.com/1038lab/ComfyUI-RMBG"),
    ("ComfyUI-Unload-Model",           "https://github.com/SeanScripts/ComfyUI-Unload-Model"),
    ("ComfyUI-Image-Filters",          "https://github.com/spacepxl/ComfyUI-Image-Filters"),
    ("comfy_mtb",                       "https://github.com/melMass/comfy_mtb"),
    ("cg-use-everywhere",              "https://github.com/chrisgoringe/cg-use-everywhere"),
]


# ─── INSTALLATION LOGIC ───
print("🚀 Starting Custom Node Installation...\n")

for name, url in CUSTOM_NODES:
    path = os.path.join(CUSTOM_NODES_DIR, name)

    # 1. FIX BROKEN DOWNLOADS
    # If folder exists but isn't a git repo, it's a failed download from a previous run.
    if os.path.exists(path) and not os.path.exists(os.path.join(path, ".git")):
        print(f"⚠️ Found broken folder '{name}', cleaning up...")
        shutil.rmtree(path)

    # 2. CLONE REPO
    if not os.path.exists(path):
        print(f"📥 Cloning: {name}")
        # We use ! instead of os.system to see real-time progress and errors
        !git clone {url} {path}
    else:
        print(f"⏭️  Already exists: {name}")
        # Optional: update existing nodes
        # !cd {path} && git pull

    # 3. INSTALL REQUIREMENTS
    req_file = os.path.join(path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"📦 Installing dependencies for {name}...")
        # We removed -q so you can see if a specific library fails to install
        !uv pip install --system -r {req_file}
    else:
        print(f"ℹ️  No requirements.txt for {name}")

print("\n✨ All custom nodes processed.")

🚀 Starting Custom Node Installation...

⏭️  Already exists: ComfyUI-Manager
📦 Installing dependencies for ComfyUI-Manager...
Using Python 3.12.13 environment at: /usr
Resolved 63 packages in 399ms
Prepared 8 packages in 385ms
Installed 8 packages in 32ms
 + aiohttp-socks==0.11.0
 + matrix-nio==0.25.2
 + pycryptodome==3.23.0
 + pygithub==2.9.0
 + pynacl==1.6.2
 + python-socks==2.8.1
 + unpaddedbase64==2.1.0
 + uv==0.11.3
📥 Cloning: rgthree-comfy
Cloning into '/content/ComfyUI/custom_nodes/rgthree-comfy'...
remote: Enumerating objects: 4678, done.
remote: Counting objects: 100% (1590/1590), done.
remote: Compressing objects: 100% (416/416), done.
remote: Total 4678 (delta 1334), reused 1174 (delta 1174), pack-reused 3088 (from 3)
Receiving objects: 100% (4678/4678), 4.95 MiB | 37.02 MiB/s, done.
Resolving deltas: 100% (3471/3471), done.
📦 Installing dependencies for rgthree-comfy...
Using Python 3.12.13 environment at: /usr
Checked in 95ms
📥 Cloning: ComfyUI-Hunyuan3DWrapper
Cloning into

In [4]:
# ── Build custom_rasterizer & make it globally importable ────
import os, subprocess, sys

RASTERIZER_SRC = "/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/hy3dgen/texgen/custom_rasterizer"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
os.environ["FORCE_CUDA"] = "1"

# Step 1: Build inplace
print("🔨 Building custom_rasterizer inplace...")
result = subprocess.run(
    [sys.executable, "-c", """
import torch.utils.cpp_extension as cpp_ext
cpp_ext._check_cuda_version = lambda *args, **kwargs: None
import os, sys
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
os.environ["FORCE_CUDA"] = "1"
sys.argv = ["setup.py", "build_ext", "--inplace"]
exec(open("setup.py").read())
"""],
    cwd=RASTERIZER_SRC,
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Build succeeded!")
else:
    print("❌ Build failed:")
    print(result.stderr[-2000:])
    raise RuntimeError("custom_rasterizer build failed")

# Step 2: Symlink BOTH the .so AND the package dir into site-packages
import glob, site

site_dir = site.getsitepackages()[0]

# 2a. Symlink the compiled .so (custom_rasterizer_kernel)
so_files = glob.glob(os.path.join(RASTERIZER_SRC, "*.so"))
for so in so_files:
    dest = os.path.join(site_dir, os.path.basename(so))
    if os.path.exists(dest) or os.path.islink(dest):
        os.remove(dest)
    os.symlink(so, dest)
    print(f"🔗 Linked: {os.path.basename(so)} → {site_dir}")

# 2b. Symlink the custom_rasterizer package dir (has __init__.py that imports the .so)
pkg_dir = os.path.join(RASTERIZER_SRC, "custom_rasterizer")
if os.path.isdir(pkg_dir):
    dest_pkg = os.path.join(site_dir, "custom_rasterizer")
    if os.path.exists(dest_pkg) or os.path.islink(dest_pkg):
        if os.path.islink(dest_pkg):
            os.remove(dest_pkg)
        else:
            import shutil
            shutil.rmtree(dest_pkg)
    os.symlink(pkg_dir, dest_pkg)
    print(f"🔗 Linked: custom_rasterizer/ → {site_dir}")

# Step 3: Verify
import importlib
custom_rasterizer = importlib.import_module("custom_rasterizer")
print(f"✅ custom_rasterizer importable globally! ({custom_rasterizer.__file__})")


🔨 Building custom_rasterizer inplace...
✅ Build succeeded!
🔗 Linked: custom_rasterizer_kernel.cpython-312-x86_64-linux-gnu.so → /usr/local/lib/python3.12/dist-packages
🔗 Linked: custom_rasterizer/ → /usr/local/lib/python3.12/dist-packages
✅ custom_rasterizer importable globally! (/usr/local/lib/python3.12/dist-packages/custom_rasterizer/__init__.py)


In [5]:
# @title 📥 Download Models { display-mode: "form" }
# @markdown Add as many models as you want in the list below.<br>
# @markdown - For **Hugging Face** → use `"type": "hf"` + `repo_id` + optional `filename` (None = full repo).<br>
# @markdown - For **Civitai / mirrors / other** → use `"type": "url"` + direct `url` + `filename`.

import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download
from tqdm import tqdm
import subprocess

# Install aria2c (required for fast URL downloads)
!apt-get update -qq && apt-get install -y -qq aria2
print("✅ aria2c installed\n")

# ────────────────────────────────────────────────
#   ↓↓↓  ADD / EDIT YOUR MODELS HERE  ↓↓↓
# ────────────────────────────────────────────────

downloads = [
    # ═══════════════════════════════════════════
    #  CHECKPOINT
    # ═══════════════════════════════════════════
    {
        "type": "hf",
        "repo_id": "misri/juggernautXL_juggXIByRundiffusion",
        "filename": "juggernautXL_juggXIByRundiffusion.safetensors",
        "subfolder": "",
        "folder": "models/checkpoints",
    },

    # ═══════════════════════════════════════════
    #  LORA — texture synthesis for 3D models
    # ═══════════════════════════════════════════
    {
        "type": "hf",
        "repo_id": "dog-god/texture-synthesis-sdxl-lora",
        "filename": "texture-synthesis-3d-base-condensed.safetensors",
        "subfolder": "",
        "folder": "models/loras",
    },

    # ═══════════════════════════════════════════
    #  CONTROLNET — Union SDXL ProMax
    # ═══════════════════════════════════════════
    {
        "type": "hf",
        "repo_id": "xinsir/controlnet-union-sdxl-1.0",
        "filename": "diffusion_pytorch_model_promax.safetensors",
        "subfolder": "",
        "folder": "models/controlnet",
    },

    # ═══════════════════════════════════════════
    #  UPSCALE MODEL — 4x ClearReality
    # ═══════════════════════════════════════════
    {
        "type": "url",
        "url": "https://huggingface.co/skbhadra/ClearRealityV1/resolve/main/4x-ClearRealityV1.pth",
        "filename": "4x-ClearRealityV1.pth",
        "folder": "models/upscale_models",
    },

    # ═══════════════════════════════════════════
    #  IP-ADAPTER — SD1.5 Plus (used by this workflow)
    # ═══════════════════════════════════════════
    {
        "type": "url",
        "url": "https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors",
        "filename": "ip-adapter-plus_sd15.safetensors",
        "folder": "models/ipadapter",
    },

    # ═══════════════════════════════════════════
    #  CLIP VISION — both needed for IP-Adapter
    # ═══════════════════════════════════════════
    {
        "type": "url",
        "url": "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors",
        "filename": "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors",
        "folder": "models/clip_vision",
    },
    {
        "type": "url",
        "url": "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors",
        "filename": "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
        "folder": "models/clip_vision",
    },

    # ═══════════════════════════════════════════
    #  FACE DETECTOR — YOLOv8 for FaceDetailer
    # ═══════════════════════════════════════════
    {
        "type": "url",
        "url": "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt",
        "filename": "face_yolov8m.pt",
        "folder": "models/ultralytics/bbox",
    },

    # ═══════════════════════════════════════════
    #  InvSR — detail restoration model
    #  NOTE: workflow uses _diftune variant
    # ═══════════════════════════════════════════
    {
        "type": "hf",
        "repo_id": "OAOA/InvSR",
        "filename": "noise_predictor_sd_turbo_v5_diftune.pth",
        "subfolder": "",
        "folder": "models/invsr",
    },

    # stabilityai/sd-turbo REMOVED — not needed by this workflow
]


# ────────────────────────────────────────────────
#     Usually no need to change anything below
# ────────────────────────────────────────────────

base_dir = "/content/ComfyUI"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def hf_download_flat(repo, filename, subfolder, target_folder):
    """Download from HF and move file flat into target_folder, ignoring subfolder structure."""
    downloaded = hf_hub_download(
        repo_id=repo,
        filename=filename,
        subfolder=subfolder or "",
        local_dir=target_folder,
    )
    final_path = os.path.join(target_folder, os.path.basename(downloaded))
    if os.path.abspath(downloaded) != os.path.abspath(final_path):
        shutil.move(downloaded, final_path)
        # Clean up empty leftover subfolders
        try:
            leftover = os.path.dirname(downloaded)
            while leftover != target_folder:
                if not os.listdir(leftover):
                    os.rmdir(leftover)
                leftover = os.path.dirname(leftover)
        except:
            pass
        print(f"   ✅ Saved: {final_path}")
    else:
        print(f"   ✅ Saved: {final_path}")

for item in tqdm(downloads, desc="Downloading models"):
    target_folder = os.path.join(base_dir, item["folder"].lstrip("/"))
    os.makedirs(target_folder, exist_ok=True)

    if item["type"].lower() in ["hf", "huggingface"]:
        repo = item["repo_id"]
        file = item.get("filename")
        subfolder = item.get("subfolder", "")
        print(f"\n📥 HF   → {repo}  →  {file or 'FULL REPO'}")

        try:
            if file is None:
                snapshot_download(
                    repo_id=repo,
                    local_dir=target_folder,
                    local_dir_use_symlinks=False,
                )
            else:
                # Check flat destination first — skip if already there
                final_path = os.path.join(target_folder, os.path.basename(file))
                if os.path.exists(final_path):
                    print(f"   ⏭️  Already exists, skipping: {final_path}")
                else:
                    hf_download_flat(repo, file, subfolder, target_folder)
        except Exception as e:
            print(f"  ⚠️ Error: {e}")
            print("   → Check repo_id / filename / subfolder")

    elif item["type"].lower() == "url":
        url   = item["url"]
        fname = item["filename"]
        print(f"\n📥 URL  → {url}  →  {fname}")
        cmd = [
            "aria2c", "--console-log-level=error", "-c",
            "-x", "16", "-s", "16", "-j", "8", "-k", "1M",
            url, "-d", target_folder, "-o", fname
        ]
        try:
            subprocess.run(cmd, check=True)
            print(f"   ✅ Saved: {os.path.join(target_folder, fname)}")
        except Exception as e:
            print(f"  ⚠️ aria2c failed: {e}")

    else:
        print(f"⚠️ Unknown type '{item['type']}' — skipping")

print("\n✅ All downloads finished! Check folders in /content/ComfyUI/models/")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/l


📥 HF   → misri/juggernautXL_juggXIByRundiffusion  →  juggernautXL_juggXIByRundiffusion.safetensors


juggernautXL_juggXIByRundiffusion.safete(…):   0%|          | 0.00/7.11G [00:00<?, ?B/s]

   ✅ Saved: /content/ComfyUI/models/checkpoints/juggernautXL_juggXIByRundiffusion.safetensors

📥 HF   → dog-god/texture-synthesis-sdxl-lora  →  texture-synthesis-3d-base-condensed.safetensors


texture-synthesis-3d-base-condensed.safe(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

...

   ✅ Saved: /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

📥 URL  → https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors  →  CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors


   ✅ Saved: /content/ComfyUI/models/clip_vision/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors

📥 URL  → https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt  →  face_yolov8m.pt


   ✅ Saved: /content/ComfyUI/models/ultralytics/bbox/face_yolov8m.pt

📥 HF   → OAOA/InvSR  →  noise_predictor_sd_turbo_v5_diftune.pth


noise_predictor_sd_turbo_v5_diftune.pth:   0%|          | 0.00/135M [00:00<?, ?B/s]

   ✅ Saved: /content/ComfyUI/models/invsr/noise_predictor_sd_turbo_v5_diftune.pth

✅ All downloads finished! Check folders in /content/ComfyUI/models/


In [ ]:
# @title 🚀 Launch ComfyUI { display-mode: "form" }
# @markdown ## 5 · Launch ComfyUI
# @markdown
# @markdown Choose launch mode and configure the batch watcher below.
# @markdown
# @markdown **Launch modes:**
# @markdown - `window` — opens ComfyUI in a new browser tab (default)
# @markdown - `iframe` — embeds ComfyUI inside the notebook cell
# @markdown - `cloudflare` — public URL via Cloudflare tunnel (share across devices)
# @markdown
# @markdown **Batch watcher** monitors an input folder and auto-runs a workflow on every dropped file.
# @markdown Leave `WORKFLOW_FILENAME` empty to just launch ComfyUI without the watcher.
import os, time, json, shutil, threading, socket, glob, requests, uuid, subprocess
from pathlib import Path
from datetime import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import builtins
if getattr(builtins, "_comfyui_launched", False):
    print("⚠️  Already running. Use Runtime → Restart to relaunch.")
    raise SystemExit()
builtins._comfyui_launched = True

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                              ║
# ╚══════════════════════════════════════════════════════════════════╝

LAUNCH_MODE  = "ngrok"  # @param ["window", "iframe", "cloudflare", "ngrok"]

# Full absolute path — examples:
# My Drive    : /content/drive/MyDrive/comfyui/image-upscale
# Shared Drive: /content/drive/Shareddrives/Figuro/image-upscale
DRIVE_BASE_PATH      = "/content/drive/Shareddrives/Figuro/image-upscale"  # @param {type:"string"}
EXTRA_OUTPUT_PATH    = ""                                        # @param {type:"string"}
WORKFLOW_FILENAME    = ""                                        # @param {type:"string"}
INPUT_NODE_ID        = ""                                        # @param {type:"string"}
POLL_INTERVAL        = 10                                        # @param {type:"integer"}
SUPPORTED_EXTENSIONS = "png,jpg,jpeg,webp,mp4,gif,bmp,tiff"    # @param {type:"string"}
NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")

# Auto-disable batch mode if no workflow is configured
BATCH_MODE = bool(WORKFLOW_FILENAME.strip())

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍  PATH RESOLUTION (works with Shared Drives + MyDrive)      ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive as _drive
if not os.path.exists("/content/drive"):
    _drive.mount('/content/drive', force_remount=True)

def validate_path(label: str, path: str) -> str:
    path = path.strip().rstrip("/")
    if not path.startswith("/content/drive/"):
        raise ValueError(f"❌ {label} must start with /content/drive/MyDrive/... or /content/drive/Shareddrives/<name>/...\n   Got: '{path}'")
    os.makedirs(path, exist_ok=True)
    return path

def safe_makedirs(path: str) -> str:
    """Create a subdir inside an existing Drive folder. Falls back to /tmp on error."""
    try:
        os.makedirs(path, exist_ok=True)
        return path
    except OSError as e:
        fallback = path.replace("/content/drive", "/tmp/drive_mirror")
        os.makedirs(fallback, exist_ok=True)
        print(f"⚠️  Cannot create '{path}': {e}")
        print(f"   → Using local fallback: '{fallback}'")
        print(f"   → Fix: create the folder manually in Drive, then re-run.")
        return fallback

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧  SETUP                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

COMFYUI_URL  = "http://127.0.0.1:8188"
EXTENSIONS   = {e.strip().lower().lstrip('.') for e in SUPPORTED_EXTENSIONS.split(',') if e.strip()}
_FILE_FIELDS = ["image","video","audio","file","mask",
                "image_path","video_path","file_path","input","source","path"]

if BATCH_MODE:
    MAIN_FOLDER   = validate_path("DRIVE_BASE_PATH", DRIVE_BASE_PATH)
    INPUT_FOLDER  = safe_makedirs(os.path.join(MAIN_FOLDER, "input"))
    PROCESSED_DIR = safe_makedirs(os.path.join(MAIN_FOLDER, "processed"))
    OUTPUT_DIR    = safe_makedirs(os.path.join(MAIN_FOLDER, "output"))
    WORKFLOW_PATH = os.path.join(MAIN_FOLDER, WORKFLOW_FILENAME)
    EXTRA_DIR     = None
    if EXTRA_OUTPUT_PATH.strip():
        EXTRA_DIR = validate_path("EXTRA_OUTPUT_PATH", EXTRA_OUTPUT_PATH.strip())
    print(f"✅ Paths ready:")
    print(f"   Input     : {INPUT_FOLDER}")
    print(f"   Processed : {PROCESSED_DIR}")
    print(f"   Output    : {OUTPUT_DIR}")
    print(f"   Workflow  : {WORKFLOW_PATH}")
else:
    MAIN_FOLDER = WORKFLOW_PATH = INPUT_FOLDER = PROCESSED_DIR = OUTPUT_DIR = EXTRA_DIR = None
    print("ℹ️  No workflow set — launching ComfyUI in manual mode (batch watcher disabled).")

# ── Batch helpers ─────────────────────────────────────────────────

def load_workflow():
    if not os.path.exists(WORKFLOW_PATH):
        raise FileNotFoundError(f"Workflow not found: {WORKFLOW_PATH}")
    with open(WORKFLOW_PATH) as f:
        return json.load(f)

def auto_detect_field(workflow, node_id):
    if node_id not in workflow:
        raise KeyError(f"Node '{node_id}' not found. Available: {list(workflow.keys())}")
    inputs = workflow[node_id].get("inputs", {})
    for known in _FILE_FIELDS:
        if known in inputs:
            return known
    media_exts = {"png","jpg","jpeg","webp","gif","bmp","tiff","tif",
                  "mp4","avi","mov","mkv","webm","mp3","wav","flac","ogg"}
    for field, val in inputs.items():
        if isinstance(val, str) and Path(val).suffix.lower().lstrip('.') in media_exts:
            return field
    short = [f for f, v in inputs.items()
             if isinstance(v, str) and len(v) < 256 and not v.startswith(("http", "{"))]
    if short:
        return short[0]
    raise ValueError(f"Can't detect input field on node '{node_id}'. Inputs: {list(inputs.keys())}")

def find_input_node(wf, preferred_id):
    """Return (node_id, field). Falls back to first LoadImage node if preferred_id missing."""
    if preferred_id and preferred_id in wf:
        return preferred_id, auto_detect_field(wf, preferred_id)
    load_types = {"LoadImage", "LoadImageMask", "LoadVideo", "VHS_LoadVideo", "ImageLoader"}
    for nid, node in wf.items():
        if node.get("class_type", "") in load_types:
            field = auto_detect_field(wf, nid)
            print(f"   ℹ️  Node '{preferred_id}' not found → auto-selected node '{nid}' ({node['class_type']})")
            print(f"   ⚠️  Update INPUT_NODE_ID to '{nid}' to silence this warning.")
            return nid, field
    raise KeyError(f"Node '{preferred_id}' not found and no LoadImage node detected. Available: {list(wf.keys())}")

def wait_for_comfyui(timeout=120):
    print("⏳ Waiting for ComfyUI...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(f"{COMFYUI_URL}/system_stats", timeout=3).status_code == 200:
                print(" ✅ Ready!")
                return
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(2)
    raise TimeoutError("ComfyUI did not start in time.")

def queue_prompt(workflow):
    client_id = str(uuid.uuid4())
    r = requests.post(f"{COMFYUI_URL}/prompt",
                      json={"prompt": workflow, "client_id": client_id}, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"Prompt error: {data['error']}")
    return data["prompt_id"]

def poll_until_done(prompt_id, timeout=600):
    deadline = time.time() + timeout
    while time.time() < deadline:
        time.sleep(3)
        try:
            r = requests.get(f"{COMFYUI_URL}/history/{prompt_id}", timeout=10)
            r.raise_for_status()
            history = r.json()
        except Exception:
            continue
        if prompt_id not in history:
            continue
        outputs = []
        for _, node_out in history[prompt_id].get("outputs", {}).items():
            for _, items in node_out.items():
                if isinstance(items, list):
                    for item in items:
                        if isinstance(item, dict) and "filename" in item:
                            outputs.append(item)
        return outputs
    raise TimeoutError(f"Prompt {prompt_id} timed out after {timeout}s")

def fetch_and_save_output(file_info, dest_dirs, stem):
    params = {"filename": file_info["filename"],
              "subfolder": file_info.get("subfolder", ""),
              "type": file_info.get("type", "output")}
    r = requests.get(f"{COMFYUI_URL}/view", params=params, timeout=120)
    r.raise_for_status()
    ext  = Path(file_info["filename"]).suffix
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{stem}_{ts}{ext}"
    saved = []
    for dest in dest_dirs:
        os.makedirs(dest, exist_ok=True)
        out_path = os.path.join(dest, name)
        with open(out_path, 'wb') as f:
            f.write(r.content)
        saved.append(out_path)
    return saved

def copy_input_to_comfyui(src_path):
    comfyui_input = "/content/ComfyUI/input"
    os.makedirs(comfyui_input, exist_ok=True)
    filename = Path(src_path).name
    shutil.copy2(src_path, os.path.join(comfyui_input, filename))
    return filename

def process_file(filepath):
    stem = Path(filepath).stem
    print(f"\n{'─'*55}")
    print(f"📂 {Path(filepath).name}  [{datetime.now().strftime('%H:%M:%S')}]")
    workflow = load_workflow()
    nid, field = find_input_node(workflow, INPUT_NODE_ID)
    filename = copy_input_to_comfyui(filepath)
    workflow[nid]["inputs"][field] = filename
    print(f"   Node [{nid}].{field} → '{filename}'")
    prompt_id = queue_prompt(workflow)
    print(f"   Queued: {prompt_id}")
    print("   ⏳ Running...", end="", flush=True)
    output_files = poll_until_done(prompt_id)
    print(f" done — {len(output_files)} output(s)")
    dest_dirs = [OUTPUT_DIR] + ([EXTRA_DIR] if EXTRA_DIR else [])
    for fi in output_files:
        for p in fetch_and_save_output(fi, dest_dirs, stem):
            print(f"   💾 {p}")
    if not output_files:
        print("   ⚠️  No outputs — check workflow has a Save node.")
    dest_proc = os.path.join(PROCESSED_DIR, Path(filepath).name)
    if os.path.exists(dest_proc):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest_proc = os.path.join(PROCESSED_DIR, f"{stem}_{ts}{Path(filepath).suffix}")
    shutil.move(filepath, dest_proc)
    print(f"   📦 Moved to processed")

def run_watcher():
    wait_for_comfyui()
    try:
        wf = load_workflow()
        nid, field = find_input_node(wf, INPUT_NODE_ID)
        cls = wf[nid].get("class_type", "unknown")
        print(f"\n✅ Workflow: {WORKFLOW_PATH}")
        print(f"   Node [{nid}] {cls} → field: '{field}'")
    except Exception as e:
        print(f"\n⚠️  Workflow check failed: {e}")
        return
    print(f"\n👀 Watching '{INPUT_FOLDER}' every {POLL_INTERVAL}s")
    print("   Drop files into the input folder to trigger the workflow.\n")
    seen_errors = {}
    while True:
        try:
            candidates = sorted(set(
                f for ext in EXTENSIONS
                for f in glob.glob(os.path.join(INPUT_FOLDER, f"*.{ext}")) +
                          glob.glob(os.path.join(INPUT_FOLDER, f"*.{ext.upper()}"))
            ))
            for fp in candidates:
                if seen_errors.get(fp, 0) >= 3:
                    continue
                try:
                    process_file(fp)
                    seen_errors.pop(fp, None)
                except Exception as e:
                    seen_errors[fp] = seen_errors.get(fp, 0) + 1
                    print(f"\n❌ Error ({seen_errors[fp]}/3) — {Path(fp).name}: {e}")
                    if seen_errors[fp] >= 3:
                        print(f"   ⛔ Giving up on {Path(fp).name}")
        except Exception as e:
            print(f"⚠️  Watcher error: {e}")
        time.sleep(POLL_INTERVAL)

# ── Launch modes ──────────────────────────────────────────────────

def start_window(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            break
        sock.close()
    from google.colab import output as co
    print("\n🌐 ComfyUI is ready!")
    co.serve_kernel_port_as_window(port)

def start_iframe(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            break
        sock.close()
    from google.colab import output as co
    print("\n🌐 ComfyUI is ready!")
    co.serve_kernel_port_as_iframe(port, height=900)
    co.serve_kernel_port_as_window(port)

def start_cloudflare(port):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], capture_output=True)
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            break
        sock.close()
    print("\n🌐 Launching Cloudflare tunnel...")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    url = None
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com" in l and "http" in l:
            url = l[l.find("http"):].strip()
            if url:
                print(f"\n✅ ComfyUI URL: {url}\n")
                break

def start_ngrok(port):
    subprocess.run(["pip", "install", "-q", "pyngrok"], capture_output=True)
    from pyngrok import ngrok
    token = NGROK_TOKEN
    try:
        from google.colab import userdata
        secret = userdata.get("NGROK_TOKEN")
        if secret:
            token = secret
    except Exception:
        pass
    if not token:
        print("❌ NGROK_TOKEN not set. Add it to Colab Secrets (🔑 icon in sidebar).")
        return
    ngrok.set_auth_token(token)
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            break
        sock.close()
    tunnel = ngrok.connect(port)
    print(f"\n✅ ComfyUI URL: {tunnel.public_url}\n")

# ── Print summary & start ─────────────────────────────────────────
print("=" * 55)
print("🗂️  ComfyUI Workflow Runner")
print("=" * 55)
print(f"  Mode       : {LAUNCH_MODE} | Batch: {BATCH_MODE}")
if BATCH_MODE:
    print(f"  Input      : {INPUT_FOLDER}")
    print(f"  Output     : {OUTPUT_DIR}")
    print(f"  Workflow   : {WORKFLOW_PATH}")
    print(f"  Node ID    : {INPUT_NODE_ID or 'auto-detect'}")
    print(f"  Poll       : {POLL_INTERVAL}s")
    if EXTRA_DIR:
        print(f"  Extra out  : {EXTRA_DIR}")
print("=" * 55)

%cd /content/ComfyUI

if LAUNCH_MODE == "iframe":
    threading.Thread(target=start_iframe,  daemon=True, args=(8188,)).start()
elif LAUNCH_MODE == "cloudflare":
    threading.Thread(target=start_cloudflare, daemon=True, args=(8188,)).start()
elif LAUNCH_MODE == "ngrok":
    threading.Thread(target=start_ngrok, daemon=True, args=(8188,)).start()
else:
    threading.Thread(target=start_window, daemon=True, args=(8188,)).start()

if BATCH_MODE:
    threading.Thread(target=run_watcher, daemon=True).start()

print("\n🚀 Starting ComfyUI...\n")
!python main.py --highvram --cuda-malloc --use-sage-attention --fast --dont-print-server

ℹ️  No workflow set — launching ComfyUI in manual mode (batch watcher disabled).
🗂️  ComfyUI Workflow Runner
  Mode       : ngrok | Batch: False
/content/ComfyUI

🚀 Starting ComfyUI...

[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-04-03 12:09:22.597
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   0.0 seconds: /content/ComfyUI/custom_nodes/rgthree-comfy
   6.6 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_ro

---

## 📝 Notes

**Workflow format** — Export via ComfyUI → Settings → Dev Mode → *Save (API Format)*. Uses numeric node IDs as top-level keys.

**Input node ID** — open `workflow.json`, find your Load Image / Load Video node, its top-level key (e.g. `"12"`) is `INPUT_NODE_ID`. The input field is auto-detected and printed on startup.

**Batch mode off** — set `BATCH_MODE = False` to just launch ComfyUI without the watcher. Useful when you want to use the UI manually.

**Launch modes** — `window` opens a new tab, `iframe` embeds in the cell, `cloudflare` gives a public shareable URL.

**Error handling** — files that fail 3 times are skipped permanently so the watcher never loops.

**Live edits** — workflow JSON is re-read on every file so you can tweak it without restarting.

---
